# Intent-Driven Video Content Pipeline

Turn any video into structured content based on your **intent** — analyze speaking techniques, extract workplace rules, break down debate logic, and more.

**Features:**
- 🎯 Intent-driven: describe what you want, AI handles the rest
- 🎬 Qwen2.5-VL video analysis with intent-focused dimensions
- 📝 LLM-generated narration with adaptive narrative structure
- 🎙️ IndexTTS2 voice cloning with emotion profile biasing
- ✨ 18 fancy text effects (综艺级花字: pop_zoom, bounce, glow, shake, etc.)
- 📐 Portrait (9:16) and landscape (16:9) output
- 💾 Auto-resume from checkpoint if interrupted

**Prerequisites:**
1. Set `LLM_API_KEY` in [Colab Secrets](https://colab.research.google.com/notebooks/secrets.ipynb)
2. Upload a **reference audio** file for TTS voice cloning

**Example intents:**
- `"分析这个人的演讲技巧"` — break down a TED talk's presentation skills
- `"从这部剧提炼职场生存法则"` — extract workplace wisdom from a drama
- `"拆解说话的艺术"` — analyze the art of communication
- `"总结这个视频的核心观点"` — summarize key arguments

## 0. Mount Google Drive & Set Model Cache

All large model files (IndexTTS2, Whisper, Qwen2.5-VL) are cached on Google Drive so they persist across sessions.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

# --- Model cache on Google Drive (avoids re-downloading every session) ---
DRIVE_CACHE = "/content/drive/MyDrive/index-tts-cache"
os.makedirs(f"{DRIVE_CACHE}/hf_home", exist_ok=True)
os.makedirs(f"{DRIVE_CACHE}/torch_home", exist_ok=True)

# HuggingFace models: Whisper, Qwen2.5-VL, IndexTTS2
os.environ["HF_HOME"] = f"{DRIVE_CACHE}/hf_home"
os.environ["TORCH_HOME"] = f"{DRIVE_CACHE}/torch_home"

print(f"Model cache: {DRIVE_CACHE}")
print(f"HF_HOME:    {os.environ['HF_HOME']}")
print(f"TORCH_HOME: {os.environ['TORCH_HOME']}")

## 1. Install dependencies

In [ ]:
!nvidia-smi
import sys
print(f"Python {sys.version}")

In [ ]:
# Clone the repo
%cd /content
!git clone -b py3.12 https://github.com/deluxebear/index-tts.git 2>/dev/null || (cd /content/index-tts && git pull)
%cd /content/index-tts

In [ ]:
%cd /content/index-tts

# Uninstall conflicting Colab packages + install project deps
!pip uninstall -y tensorflow keras 2>/dev/null
!pip install ninja
# Install project + torchvision together so pip resolves matching torch/torchvision/torchaudio versions
!pip install -e ".[webui]" torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu128

# Intent pipeline deps (same as highlight + identical VL requirements)
!pip install -q whisperx soundfile openai
!pip install -q "transformers>=4.52.1" qwen-vl-utils accelerate bitsandbytes

# FlashAttention2: 2-4x faster attention, saves VRAM for video analysis
!pip install flash-attn --no-build-isolation


# Restart runtime so new packages take effect
print("Restarting runtime to apply package changes...")
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

### After runtime restart, run this cell to restore environment

The install cell above restarts the runtime. Run this cell to re-mount Drive and verify packages.

In [ ]:
import os
from google.colab import drive

# Re-mount Drive and restore env vars after runtime restart
drive.mount('/content/drive')

DRIVE_CACHE = "/content/drive/MyDrive/index-tts-cache"
os.environ["HF_HOME"] = f"{DRIVE_CACHE}/hf_home"
os.environ["TORCH_HOME"] = f"{DRIVE_CACHE}/torch_home"

%cd /content/index-tts

# Verify packages loaded correctly
import torch, numpy, numba
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()}")
print(f"numpy={numpy.__version__}")
print(f"numba={numba.__version__}")
print("All good!")

## 2. Download IndexTTS2 checkpoints

Checkpoints are downloaded to Google Drive and used directly from there.

In [ ]:
import os

DRIVE_CKPT = f"{DRIVE_CACHE}/checkpoints-2.5"
os.environ["INDEX_TTS_MODEL_DIR"] = DRIVE_CKPT

# Download to Drive (persists across sessions)
if not os.path.exists(f"{DRIVE_CKPT}/config.yaml"):
    print("Downloading IndexTTS2 checkpoints to Google Drive (first time only)...")
    !huggingface-cli download IndexTeam/IndexTTS-2.5 --local-dir "{DRIVE_CKPT}"
else:
    print(f"Checkpoints already cached at {DRIVE_CKPT}")

# Set MODEL_DIR for pipeline use
MODEL_DIR = DRIVE_CKPT
print(f"Model dir: {MODEL_DIR}")
!ls -la "{MODEL_DIR}/config.yaml"

## 3. Configuration

In [ ]:
from google.colab import userdata

LLM_API_KEY = userdata.get('LLM_API_KEY')  # LLM API key — for intent planning & script generation

# --- LLM provider (uncomment one) ---
LLM_API_BASE = "https://api.openai.com/v1"       ; LLM_MODEL = "gpt-4o-mini"
# LLM_API_BASE = "https://api.deepseek.com/v1"    ; LLM_MODEL = "deepseek-chat"
# LLM_API_BASE = "https://generativelanguage.googleapis.com/v1beta/openai/" ; LLM_MODEL = "gemini-2.0-flash"

# --- Intent pipeline settings ---
VL_MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"  # Video understanding model (~18GB VRAM)
TARGET_DURATION = 180   # Target output duration in seconds (2-5 min recommended)
ORIENTATION = "portrait" # "portrait" (1080x1920) or "landscape" (1920x1080)
USE_FP16 = True         # FP16 inference (faster, less VRAM)
CLEANUP = False         # Delete intermediate files after completion

# --- Paths ---
WORK_DIR = "/content/intent_workspace"    # Intermediate files
# MODEL_DIR is set in the checkpoints cell above

print(f"LLM: {LLM_MODEL} @ {LLM_API_BASE}")
print(f"VL Model: {VL_MODEL}")
print(f"Orientation: {ORIENTATION}")
print(f"Target duration: {TARGET_DURATION}s ({TARGET_DURATION/60:.1f}min)")
print(f"Model dir: {MODEL_DIR}")
print(f"Work dir: {WORK_DIR}")
print(f"FP16: {USE_FP16}, Cleanup: {CLEANUP}")

## 4. Pre-download heavy models (optional)

Run this cell once to download Whisper and Qwen2.5-VL models to Google Drive. Subsequent sessions will reuse cached models.

In [ ]:
# Pre-download Whisper model (~3GB)
import whisperx
print("Loading Whisper model (cached on Drive)...")
_model = whisperx.load_model("large-v2", "cuda", compute_type="float16")
del _model
print("Whisper model cached.")

# Pre-download Qwen2.5-VL model (~18GB)
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
print(f"Loading {VL_MODEL} (cached on Drive)...")
_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VL_MODEL, torch_dtype="auto", device_map="auto"
)
_processor = AutoProcessor.from_pretrained(VL_MODEL)
del _model, _processor
print("Qwen2.5-VL model cached.")

import torch; torch.cuda.empty_cache()
import gc; gc.collect()
print("\nAll models cached on Google Drive. Future sessions will start faster.")

## 5. Run: Upload video & describe your intent

**Two options:**
- **Upload** a video and reference audio directly
- **Use Google Drive paths** (uncomment the lines below)

In [ ]:
import sys, os
os.chdir('/content/index-tts')
if '/content/index-tts' not in sys.path:
    sys.path.insert(0, '/content/index-tts')

from intent_pipeline import create_intent_video
from google.colab import files
from pathlib import Path

# === Your intent (describe what you want from the video) ===
INTENT = "分析这个人的演讲技巧"  # <-- Change this!

# Examples:
# INTENT = "分析这个人的演讲技巧"
# INTENT = "从这部剧提炼职场生存法则"
# INTENT = "拆解这段对话中说话的艺术"
# INTENT = "总结这个视频的核心观点"
# INTENT = "分析这段辩论的逻辑技巧"

# === Upload video ===
print("Upload your video file:")
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

# === Upload reference audio (for TTS voice cloning) ===
print("\nUpload reference audio (5-15s WAV of target voice):")
uploaded_ref = files.upload()
ref_audio = list(uploaded_ref.keys())[0]

# === Option: Use Google Drive paths instead (uncomment below) ===
# video_path = "/content/drive/MyDrive/videos/ted_talk.mp4"
# ref_audio = "/content/drive/MyDrive/voices/narrator.wav"

# === Run (auto-resumes if interrupted) ===
stem = Path(video_path).stem
output_path = f"/content/{stem}_intent.mp4"

create_intent_video(
    video_path=video_path,
    ref_audio=ref_audio,
    intent=INTENT,
    output_path=output_path,
    orientation=ORIENTATION,
    work_dir=WORK_DIR,
    llm_api_key=LLM_API_KEY,
    llm_api_base=LLM_API_BASE,
    llm_model=LLM_MODEL,
    vl_model=VL_MODEL,
    model_dir=MODEL_DIR,
    target_duration=TARGET_DURATION,
    use_fp16=USE_FP16,
    cleanup=CLEANUP,
)

# Play result inline
from IPython.display import Video, display
display(Video(output_path, embed=True, width=640))

In [ ]:
# Download the result
files.download(output_path)

## 6. Batch mode (Google Drive)

Process all videos in a directory with the same intent. Output goes to a parallel folder.

In [ ]:
import sys, os
os.chdir('/content/index-tts')
if '/content/index-tts' not in sys.path:
    sys.path.insert(0, '/content/index-tts')

from intent_pipeline import intent_batch

BATCH_INTENT = "分析演讲技巧"  # <-- Your intent for all videos

INPUT_DIR  = "/content/drive/MyDrive/videos/input"         # <- your input folder
OUTPUT_DIR = "/content/drive/MyDrive/videos/intent_output"  # <- output folder
REF_AUDIO  = "/content/drive/MyDrive/voices/narrator.wav"   # <- reference voice

intent_batch(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    ref_audio=REF_AUDIO,
    intent=BATCH_INTENT,
    orientation=ORIENTATION,
    work_dir=WORK_DIR,
    llm_api_key=LLM_API_KEY,
    llm_api_base=LLM_API_BASE,
    llm_model=LLM_MODEL,
    vl_model=VL_MODEL,
    model_dir=MODEL_DIR,
    target_duration=TARGET_DURATION,
    use_fp16=USE_FP16,
    cleanup=CLEANUP,
)
print(f"\nAll done! Check: {OUTPUT_DIR}")

---
## 7. Inspect intermediate results

The pipeline saves intermediate files for debugging. Check the `intent_workspace/` directory.

In [ ]:
import json, os
from IPython.display import Audio, display

# Find the most recent work directory
work_dirs = sorted(
    [d for d in os.listdir(WORK_DIR) if os.path.isdir(f"{WORK_DIR}/{d}")]
) if os.path.isdir(WORK_DIR) else []

if work_dirs:
    work_dir = f"{WORK_DIR}/{work_dirs[-1]}"
    print(f"Work directory: {work_dir}")
    print(f"Contents: {os.listdir(work_dir)}")

    # Show intent plan
    ckpt_path = f"{work_dir}/intent_checkpoint.json"
    if os.path.exists(ckpt_path):
        with open(ckpt_path) as f:
            ckpt = json.load(f)
        plan = ckpt.get('intent_plan', {})
        if plan:
            print(f"\n--- Intent Plan ---")
            print(f"  Summary:   {plan.get('intent_summary', '')}")
            print(f"  Style:     {plan.get('content_style', '')}")
            print(f"  Persona:   {plan.get('persona', '')}")
            print(f"  Structure: {plan.get('narrative_structure', '')}")
            print(f"  Tone:      {', '.join(plan.get('tone_keywords', []))}")
            print(f"  Title:     {plan.get('title_suggestion', '')}")
            dims = plan.get('analysis_dimensions', [])
            print(f"  Dimensions: {', '.join(dims)}")

    # Show transcript
    transcript_path = f"{work_dir}/transcript.json"
    if os.path.exists(transcript_path):
        with open(transcript_path) as f:
            segments = json.load(f)
        print(f"\n--- Transcript ({len(segments)} segments) ---")
        for s in segments[:10]:
            print(f"  {s['start']:.1f}-{s['end']:.1f}s: {s['text']}")
        if len(segments) > 10:
            print(f"  ... and {len(segments)-10} more")

    # Show VL analysis
    analysis_path = f"{work_dir}/intent_video_analysis.json"
    if os.path.exists(analysis_path):
        with open(analysis_path) as f:
            analysis = json.load(f)
        events = analysis.get('events', [])
        print(f"\n--- Intent-Focused VL Analysis ({len(events)} events) ---")
        print(f"  Type: {analysis.get('video_type', '')}")
        print(f"  Summary: {analysis.get('overall_summary', '')}")
        findings = analysis.get('intent_relevant_findings', '')
        if findings:
            print(f"  Intent findings: {findings[:200]}")
        for e in events[:10]:
            intent_score = e.get('intent_score', '-')
            print(f"  [{e.get('start_time',0)}-{e.get('end_time',0)}s] "
                  f"imp={e.get('importance',0)} intent={intent_score} "
                  f"{e.get('description','')[:50]}")

    # Show script with fancy texts
    script_path = f"{work_dir}/intent_script.json"
    if os.path.exists(script_path):
        with open(script_path) as f:
            script = json.load(f)
        print(f"\n--- Intent Script ({len(script)} segments) ---")
        for i, seg in enumerate(script):
            narration = seg.get('narration', '')
            emo = seg.get('emo_vector', [])
            fancy = seg.get('fancy_texts', [])
            print(f"  #{i} [{seg.get('start_time',0):.1f}-{seg.get('end_time',0):.1f}s] "
                  f"{narration[:50]}...")
            if emo:
                dims = ['happy','angry','sad','afraid','disgusted','melancholic','surprised','calm']
                top = sorted(zip(dims, emo), key=lambda x: -x[1])[:3]
                print(f"       emotion: {', '.join(f'{k}={v:.2f}' for k,v in top)}")
            if fancy:
                print(f"       fancy_texts: {len(fancy)} items")
                for ft in fancy[:3]:
                    print(f"         - [{ft.get('effect','')}] "
                          f"pos={ft.get('position',[])} "
                          f"{ft.get('text','')[:30]}")

    # Play narration audio files
    tts_dir = f"{work_dir}/tts_output"
    if os.path.isdir(tts_dir):
        tts_files = sorted(f for f in os.listdir(tts_dir) if f.endswith('.wav'))[:3]
        if tts_files:
            print(f"\n--- Narration samples ({len(tts_files)} shown) ---")
            for f in tts_files:
                print(f"  {f}")
                display(Audio(f"{tts_dir}/{f}"))

    # Show fancy ASS file preview
    ass_path = f"{work_dir}/fancy_subtitles.ass"
    if os.path.exists(ass_path):
        with open(ass_path, 'r') as f:
            ass_content = f.read()
        dialogue_lines = [l for l in ass_content.split('\n') if l.startswith('Dialogue:')]
        print(f"\n--- Fancy ASS ({len(dialogue_lines)} dialogue lines) ---")
        for l in dialogue_lines[:10]:
            print(f"  {l[:120]}..." if len(l) > 120 else f"  {l}")
        if len(dialogue_lines) > 10:
            print(f"  ... and {len(dialogue_lines)-10} more")
else:
    print("No work directories found. Run the pipeline first.")